# 15 · Detailed Scores & Cross-Platform Generalisation

**Goal.** Report the full per-class scores and the headline finding: **how well a model trained on one
platform works on completely different platforms**.

**Why it matters.** Real deployment means new platforms with new writing styles. The gap between
in-platform and cross-platform performance is the most honest measure of usefulness.

**What happens below.** Per-tier precision/recall/F1 with confidence intervals, the cross-platform
result, and a consolidated table of all ablations.

**Output:** `data/stage5_validation/validation_report.json`.

## 0. Environment

In [1]:
import os, sys, json, time, math, logging, warnings, random
from pathlib import Path
from collections import Counter, defaultdict

for _stream in ("stdout", "stderr"):
    try:
        getattr(sys, _stream).reconfigure(encoding="utf-8")
    except Exception:
        pass
import numpy as np
def _try(n):
    try:
        __import__(n); return True
    except Exception:
        return False
assert _try("torch") and _try("sklearn"), "notebook 15 needs torch + sklearn"
HAS_SCIPY = _try("scipy"); HAS_MPL = _try("matplotlib")
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("val")
PART_VERSION = "1.0"
SEEDS = [42, 2023, 2024, 0, 7]

## 1. Config, paths, model + losses, cached data (reused verbatim from notebook 14/3.2)

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F

def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data" / "stage3_model").exists() or (d / "data" / "stage2_semantic").exists() or (d / ".env").exists():
            return d
    return here

PROJECT_ROOT = find_project_root()
SEM_DIR = PROJECT_ROOT / "data" / "stage2_semantic"
TEMP_DIR = PROJECT_ROOT / "data" / "stage1_temporal"
MODEL_DIR = PROJECT_ROOT / "data" / "stage3_model"
OUT_DIR = PROJECT_ROOT / "data" / "stage5_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = json.loads((MODEL_DIR / "model_config.json").read_text(encoding="utf-8"))
FUSION_DIM = CFG["fusion_dim"]; PRIM_CAPS = CFG["prim_caps"]; PRIM_DIM = CFG["prim_dim"]
CLASS_DIM = CFG["class_dim"]; N_CLASSES = 4
ROUTING = CFG["routing"]; GATE = CFG["gate"]; DROPOUT = 0.3; TEMP_FEATS = CFG["temp_feats"]
EPOCHS = CFG["epochs"]; LR = CFG["lr"]; BATCH = CFG["batch"]
FOCAL_GAMMA = CFG["focal_gamma"]; CONTRAST_W = CFG["contrast_w"]; CONTRAST_T = 0.1; WEIGHT_DECAY = 1e-4
DEVICE = "cpu"; TIER_NAMES = {0: "Neutral", 1: "Stress", 2: "Moderate", 3: "Crisis"}

def squash(s, dim=-1, eps=1e-8):
    n2 = (s * s).sum(dim=dim, keepdim=True)
    return (n2 / (1.0 + n2)) * (s / torch.sqrt(n2 + eps))
class CapsuleRouting(nn.Module):
    def __init__(self, n_in, d_in, n_out, d_out, routing="static", iters=3):
        super().__init__(); self.n_in, self.n_out, self.routing, self.iters = n_in, n_out, routing, iters
        self.W = nn.Parameter(0.01 * torch.randn(1, n_in, n_out, d_out, d_in)); self.b_static = nn.Parameter(torch.zeros(1, n_in, n_out))
    def forward(self, u):
        u_hat = torch.matmul(self.W, u.unsqueeze(2).unsqueeze(-1)).squeeze(-1)
        c = torch.softmax(self.b_static, dim=2).unsqueeze(-1); return squash((c * u_hat).sum(dim=1))
class DriftCapsNet(nn.Module):
    def __init__(self, text_dim, emo_dim, temp_dim=6, d=FUSION_DIM, gate=GATE, fusion="attention", routing=ROUTING, dropout=DROPOUT):
        super().__init__(); self.gate_kind, self.fusion = gate, fusion
        self.text_proj = nn.Linear(text_dim, PRIM_CAPS * PRIM_DIM); self.gate_lin = nn.Linear(PRIM_CAPS * PRIM_DIM, PRIM_CAPS * PRIM_DIM)
        self.caps = CapsuleRouting(PRIM_CAPS, PRIM_DIM, N_CLASSES, CLASS_DIM, routing=routing)
        self.q_proj = nn.Linear(N_CLASSES * CLASS_DIM, d); self.e_proj = nn.Linear(emo_dim, d)
        self.t_proj = nn.Sequential(nn.Linear(temp_dim, 64), nn.ReLU(), nn.Linear(64, d))
        self.drop = nn.Dropout(dropout); self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d if fusion in ("attention", "text_only") else 3 * d, N_CLASSES); self.d = d
    def gate(self, H):
        if self.gate_kind == "none": return H
        if self.gate_kind == "sigmoid": return H * torch.sigmoid(self.gate_lin(H))
        return H * (F.elu(self.gate_lin(H)) + 1.0)
    def temporal_rep(self, x_t, mask): return self.t_proj(x_t) * mask.unsqueeze(-1)
    def forward(self, x_text, x_emo, x_temp, mask, return_z=False):
        H = self.drop(F.elu(self.text_proj(x_text))); U = squash(self.gate(H).view(-1, PRIM_CAPS, PRIM_DIM))
        V = self.caps(U).reshape(x_text.size(0), -1); q = self.q_proj(V); e = self.e_proj(x_emo); tau = self.temporal_rep(x_temp, mask)
        if self.fusion == "attention":
            kv = torch.stack([tau, e], dim=1)
            att = torch.softmax(torch.matmul(q.unsqueeze(1), kv.transpose(1, 2)) / math.sqrt(self.d), dim=-1)
            z = self.norm(q + torch.matmul(att, kv).squeeze(1))
        elif self.fusion == "concat": z = torch.cat([q, tau, e], dim=-1)
        else: z = q
        logits = self.head(z); return (logits, F.normalize(z, dim=-1)) if return_z else logits
class FocalLoss(nn.Module):
    def __init__(self, weight, gamma=FOCAL_GAMMA): super().__init__(); self.weight = weight; self.gamma = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, -1); p = logp.exp(); w = self.weight[y]
        pt = p.gather(1, y.unsqueeze(1)).squeeze(1).clamp(1e-6, 1.0); logpt = logp.gather(1, y.unsqueeze(1)).squeeze(1)
        return (-w * (1 - pt) ** self.gamma * logpt).mean()
def supervised_contrastive(z, y, temp=CONTRAST_T):
    sim = torch.matmul(z, z.t()) / temp; B = z.size(0); lm = ~torch.eye(B, dtype=torch.bool, device=z.device)
    pos = (y.unsqueeze(0) == y.unsqueeze(1)) & lm; sim = sim.masked_fill(~lm, -1e9)
    logp = sim - torch.logsumexp(sim, 1, keepdim=True); pc = pos.sum(1); valid = pc > 0
    if valid.sum() == 0: return torch.tensor(0.0, device=z.device)
    return -((logp.masked_fill(~pos, 0.0).sum(1)[valid]) / pc[valid].clamp(min=1)).mean()

def load_jsonl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
train_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_train.jsonl")
test_rows = load_jsonl(SEM_DIR / "corpus_preprocessed_test.jsonl")
assert set(r["platform"] for r in test_rows) <= {"reddit", "twitter"}, "TEST leak!"
def muril_matrix(split, rows):
    emb = np.load(SEM_DIR / f"embeddings_{split}.npy").astype(np.float32)
    idx = json.loads((SEM_DIR / f"embeddings_index_{split}.json").read_text(encoding="utf-8"))
    id2 = {rid: i for i, rid in enumerate(idx)}; return np.vstack([emb[id2[r["record_id"]]] for r in rows])
MURIL_TR = muril_matrix("train", train_rows); MURIL_TE = muril_matrix("test", test_rows)
EMO_TR = np.load(MODEL_DIR / "emo_embeddings_train.npy").astype(np.float32)
EMO_TE = np.load(MODEL_DIR / "emo_embeddings_test.npy").astype(np.float32)
def load_temporal(s): return {r["record_id"]: r for r in load_jsonl(TEMP_DIR / f"temporal_features_{s}.jsonl")}
tt, te_ = load_temporal("train"), load_temporal("test")
def tmat(rows, by):
    f, m = [], []
    for r in rows:
        t = by.get(r["record_id"], {}); f.append([float(t.get(k, 0.0)) for k in TEMP_FEATS])
        m.append(float(t.get("temporal_mask", r.get("temporal_mask", 1))))
    return np.array(f, np.float32), np.array(m, np.float32)
TEMP_TR, MASK_TR = tmat(train_rows, tt); TEMP_TE, MASK_TE = tmat(test_rows, te_)
y_train = np.array([int(r["tier"]) for r in train_rows]); y_test = np.array([int(r["tier"]) for r in test_rows])
PLATFORM = np.array([r["platform"] for r in test_rows])
log.info("loaded: train=%d test=%d (reddit=%d twitter=%d)", len(train_rows), len(test_rows),
         int((PLATFORM=='reddit').sum()), int((PLATFORM=='twitter').sum()))

2026-08-08 21:11:54,680 | INFO | loaded: train=1971 test=2768 (reddit=1302 twitter=1466)


## 2. Train / predict drivers + CI helper (seeded; backbones frozen)

In [3]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix, f1_score,
                             matthews_corrcoef, accuracy_score, roc_auc_score, recall_score)

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
def to_t(*a): return [torch.tensor(x) for x in a]

def train_one(idx, fusion, seed):
    set_seed(seed)
    Xt, Xe, Xtau, M, Y = to_t(MURIL_TR[idx], EMO_TR[idx], TEMP_TR[idx], MASK_TR[idx], y_train[idx])
    ds = TensorDataset(Xt, Xe, Xtau, M, Y.long()); g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(ds, batch_size=BATCH, shuffle=True, generator=g)
    counts = np.bincount(y_train[idx], minlength=4).astype(float)
    cls_w = torch.tensor(counts.sum() / (4.0 * np.clip(counts, 1, None)), dtype=torch.float)
    set_seed(seed); model = DriftCapsNet(MURIL_TR.shape[1], EMO_TR.shape[1], fusion=fusion)
    focal = FocalLoss(cls_w); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    model.train()
    for ep in range(EPOCHS):
        for xt, xe, xtau, m, yb in dl:
            opt.zero_grad(); logits, z = model(xt, xe, xtau, m, return_z=True)
            (focal(logits, yb) + CONTRAST_W * supervised_contrastive(z, yb)).backward(); opt.step()
    return model

@torch.no_grad()
def predict(model, Xt, Xe, Xtau, M):
    model.eval(); logits = model(torch.tensor(Xt), torch.tensor(Xe), torch.tensor(Xtau), torch.tensor(M))
    prob = torch.softmax(logits, -1).numpy(); return prob.argmax(1), prob

def mean_ci(vals, conf=0.95):
    vals = [v for v in vals if v is not None]; n = len(vals)
    if n == 0: return None
    m = float(np.mean(vals)); sd = float(np.std(vals, ddof=1)) if n > 1 else 0.0; sem = sd / math.sqrt(n) if n > 1 else 0.0
    if HAS_SCIPY and n > 1:
        from scipy.stats import t as td; h = float(td.ppf(0.5 + conf / 2, n - 1)) * sem
    else: h = 1.96 * sem
    return {"mean": round(m, 4), "sd": round(sd, 4), "ci95_lo": round(m - h, 4), "ci95_hi": round(m + h, 4), "n": n}
log.info("drivers ready")

2026-08-08 21:11:55,820 | INFO | drivers ready


## 3. Classification suite ± CI — re-run 5 seeds, collect predictions

In [4]:
ALL = np.arange(len(train_rows))
seed_pred = {}
per_tier = {TIER_NAMES[t]: {"P": [], "R": [], "F1": []} for t in range(4)}
agg = {"accuracy": [], "macro_f1": [], "mcc": [], "roc_auc": []}
conf_stack = []
plat_metrics = {"reddit": {"accuracy": [], "macro_f1": [], "mcc": []},
                "twitter": {"accuracy": [], "macro_f1": [], "mcc": []}}
t0 = time.time()
for s in SEEDS:
    model = train_one(ALL, "attention", s)
    pred, prob = predict(model, MURIL_TE, EMO_TE, TEMP_TE, MASK_TE); seed_pred[s] = pred
    P, R, Fa, _ = precision_recall_fscore_support(y_test, pred, labels=[0,1,2,3], zero_division=0)
    for t in range(4):
        per_tier[TIER_NAMES[t]]["P"].append(float(P[t])); per_tier[TIER_NAMES[t]]["R"].append(float(R[t])); per_tier[TIER_NAMES[t]]["F1"].append(float(Fa[t]))
    agg["accuracy"].append(float(accuracy_score(y_test, pred))); agg["macro_f1"].append(float(f1_score(y_test, pred, average="macro", zero_division=0)))
    agg["mcc"].append(float(matthews_corrcoef(y_test, pred)))
    try: agg["roc_auc"].append(float(roc_auc_score(y_test, prob, multi_class="ovr", average="macro", labels=[0,1,2,3])))
    except Exception: agg["roc_auc"].append(None)
    conf_stack.append(confusion_matrix(y_test, pred, labels=[0,1,2,3]))
    for pl in ("reddit", "twitter"):
        mask = PLATFORM == pl; yt, yp = y_test[mask], pred[mask]
        plat_metrics[pl]["accuracy"].append(float(accuracy_score(yt, yp)))
        plat_metrics[pl]["macro_f1"].append(float(f1_score(yt, yp, average="macro", zero_division=0)))
        plat_metrics[pl]["mcc"].append(float(matthews_corrcoef(yt, yp)) if len(set(yt)) > 1 else None)
mean_conf = np.mean(conf_stack, axis=0)
classification = {
    "overall_ci": {k: mean_ci(v) for k, v in agg.items()},
    "per_tier_ci": {nm: {k: mean_ci(v) for k, v in d.items()} for nm, d in per_tier.items()},
    "mean_confusion_matrix": [[round(float(x), 2) for x in row] for row in mean_conf],
}
log.info("classification suite (5 seeds) in %.1fs: macroF1=%.3f MCC=%.3f",
         time.time() - t0, classification["overall_ci"]["macro_f1"]["mean"], classification["overall_ci"]["mcc"]["mean"])
for t in range(4):
    c = classification["per_tier_ci"][TIER_NAMES[t]]
    log.info("  %-9s P=%.3f R=%.3f F1=%.3f", TIER_NAMES[t], c["P"]["mean"], c["R"]["mean"], c["F1"]["mean"])

2026-08-08 21:28:29,878 | INFO | classification suite (5 seeds) in 994.0s: macroF1=0.465 MCC=0.429


2026-08-08 21:28:29,885 | INFO |   Neutral   P=0.690 R=0.937 F1=0.795


2026-08-08 21:28:29,886 | INFO |   Stress    P=0.805 R=0.358 F1=0.485


2026-08-08 21:28:29,888 | INFO |   Moderate  P=0.685 R=0.372 F1=0.475


2026-08-08 21:28:29,889 | INFO |   Crisis    P=0.147 R=0.100 F1=0.104


## 4. Cross-platform generalization — the in-domain ↔ cross-platform gap

The frozen Reddit+Twitter TEST **is** the train-YouTube/test-other-platform domain-shift result. We contrast it
with the **in-domain k-fold** (notebook 14) and break it down **per platform** as mean ± CI — noting the class-prior
confound (Reddit ≈ 90 % Neutral) already established in notebook 13.

In [5]:
kfold = json.loads((OUT_DIR / "kfold.json").read_text(encoding="utf-8"))
in_domain_f1 = kfold["ci"]["macro_f1"]["mean"]
cross_f1 = classification["overall_ci"]["macro_f1"]["mean"]
generalization = {
    "setup": "train on YouTube (in-domain), test on frozen Reddit+Twitter (cross-platform) — P3 / ",
    "in_domain_kfold_macro_f1": in_domain_f1,
    "cross_platform_test_macro_f1": cross_f1,
    "generalization_gap_macro_f1": round(in_domain_f1 - cross_f1, 4),
    "per_platform_ci": {pl: {k: mean_ci(v) for k, v in d.items()} for pl, d in plat_metrics.items()},
    "platform_class_prior": {pl: {TIER_NAMES[t]: int(((PLATFORM == pl) & (y_test == t)).sum()) for t in range(4)}
                             for pl in ("reddit", "twitter")},
    "confound_note": "Reddit ~90% Neutral vs Twitter balanced -> raw per-platform gaps partly reflect composition "
                     "(established in notebook 13); the in-domain->cross-platform gap is the headline generalization finding.",
}
print("=== CROSS-PLATFORM GENERALIZATION ===")
print("  in-domain k-fold macroF1 = %.3f  ->  cross-platform TEST macroF1 = %.3f  (gap %.3f)" % (
    in_domain_f1, cross_f1, generalization["generalization_gap_macro_f1"]))
for pl in ("reddit", "twitter"):
    d = generalization["per_platform_ci"][pl]
    print("  %-8s macroF1=%.3f [%.3f,%.3f]  MCC=%.3f" % (
        pl, d["macro_f1"]["mean"], d["macro_f1"]["ci95_lo"], d["macro_f1"]["ci95_hi"], d["mcc"]["mean"]))

=== CROSS-PLATFORM GENERALIZATION ===
  in-domain k-fold macroF1 = 0.834  ->  cross-platform TEST macroF1 = 0.465  (gap 0.369)
  reddit   macroF1=0.442 [0.411,0.472]  MCC=0.401
  twitter  macroF1=0.389 [0.367,0.411]  MCC=0.361


## 5. Augmentation on/off ablation — cheap, via cached-embedding subsetting

We isolate the augmentation contribution **without re-embedding** by training the head on cumulative `aug_method`
subsets of the *cached* TRAIN embeddings: original → +CSS → +EDA → +generation (full). Seed 42 (matching the
single-run methodology of the reused notebook 10 ablations).

In [6]:
aug_of = np.array([r.get("aug_method", "original") for r in train_rows])
def idx_where(methods): return np.array([i for i, a in enumerate(aug_of) if a in methods])
AUG_VARIANTS = [
    ("original_only", {"original"}),
    ("+CSS", {"original", "css"}),
    ("+EDA", {"original", "css", "eda_synonym", "eda_delete"}),
    ("+generation (full)", {"original", "css", "eda_synonym", "eda_delete", "generation"}),
]
aug_rows = []
for name, methods in AUG_VARIANTS:
    idx = idx_where(methods)
    model = train_one(idx, "attention", 42)
    pred, prob = predict(model, MURIL_TE, EMO_TE, TEMP_TE, MASK_TE)
    row = {"variant": name, "n_train": int(len(idx)),
           "train_tier_dist": {TIER_NAMES[t]: int((y_train[idx] == t).sum()) for t in range(4)},
           "macro_f1": round(float(f1_score(y_test, pred, average="macro", zero_division=0)), 4),
           "mcc": round(float(matthews_corrcoef(y_test, pred)), 4),
           "tier3_recall": round(float(recall_score(y_test, pred, labels=[3], average=None, zero_division=0)[0]), 4)}
    aug_rows.append(row)
    log.info("  aug %-20s n=%3d macroF1=%.3f MCC=%.3f T3rec=%.3f", name, row["n_train"], row["macro_f1"], row["mcc"], row["tier3_recall"])
aug_base = aug_rows[-1]["macro_f1"]
for r in aug_rows: r["d_macro_f1_vs_full"] = round(r["macro_f1"] - aug_base, 4)

2026-08-08 21:31:59,822 | INFO |   aug original_only        n=1846 macroF1=0.551 MCC=0.532 T3rec=0.109


2026-08-08 21:35:41,652 | INFO |   aug +CSS                 n=1897 macroF1=0.487 MCC=0.468 T3rec=0.087


2026-08-08 21:39:35,097 | INFO |   aug +EDA                 n=1930 macroF1=0.444 MCC=0.428 T3rec=0.065


2026-08-08 21:43:28,968 | INFO |   aug +generation (full)   n=1971 macroF1=0.492 MCC=0.457 T3rec=0.109


## 6. Consolidate the ablation battery + efficiency — REUSE prior reports

In [7]:
def load_json(p):
    try: return json.loads(Path(p).read_text(encoding="utf-8"))
    except Exception as e: log.warning("could not load %s: %s", p, e); return None

abl = load_json(MODEL_DIR / "ablation_report.json")          # 3.3 routing/gate/fusion/loss
bb = load_json(SEM_DIR / "backbone_benchmark.json")          # 2.3 frozen probe
bbft = load_json(SEM_DIR / "backbone_benchmark_finetuned.json")  # 2.3 fine-tuned
toks = load_json(SEM_DIR / "token_sensitivity.json")         # 2.3 token cap
eff = load_json(MODEL_DIR / "efficiency.json")               # 3.3 efficiency

def backbone_rows(report, cap="128"):
    out = []
    if not report: return out
    for bk, d in report.get("results", {}).items():
        c = d.get("by_token_cap", {}).get(cap, {})
        if c:
            out.append({"backbone": bk, "macro_f1": c.get("macro_f1"), "mcc": c.get("mcc"), "tier3_recall": c.get("tier3_recall")})
    return out

ablation_battery = {
    "architecture_3.3": {"source": "data/stage3_model/ablation_report.json (notebook 10, -)",
                         "baseline": abl.get("ablations", {}).get("baseline") if abl else None,
                         "routing": abl.get("ablations", {}).get("routing") if abl else None,
                         "gate": abl.get("ablations", {}).get("gate") if abl else None,
                         "fusion": abl.get("ablations", {}).get("fusion") if abl else None,
                         "loss": abl.get("ablations", {}).get("loss") if abl else None},
    "backbone_2.3_E2.7": {"source": "backbone_benchmark{,_finetuned}.json (notebook 07)",
                          "frozen_probe@128": backbone_rows(bb, "128"),
                          "finetuned@128": backbone_rows(bbft, "128"),
                          "note": "gated indic-bert=403 -> ungated IndicBERTv2 substituted; IndicBERTv2 best FT (MCC +0.160)."},
    "token_cap_2.3_E2.8": {"source": "token_sensitivity.json (notebook 07)", "by_backbone": toks.get("by_backbone") if toks else None},
    "augmentation_5.2_E1.3": {"source": "computed here (cached-embedding subset, seed 42)", "rows": aug_rows},
}
efficiency = {"source": "data/stage3_model/efficiency.json (notebook 10, )",
              "trainable_params_head": eff.get("trainable_params_head") if eff else None,
              "train_seconds_total": eff.get("train_seconds_total") if eff else None,
              "epoch_ms": (abl.get("ablations", {}).get("baseline", {}) if abl else {}).get("epoch_ms"),
              "infer_ms_per_sample": (abl.get("ablations", {}).get("baseline", {}) if abl else {}).get("infer_ms_per_sample"),
              "note_backbones": eff.get("note_backbones") if eff else None}
log.info("consolidated: routing/gate/fusion/loss (3.3) + backbone (2.3) + token-cap (2.3) + augmentation (new) + efficiency (3.3)")
print("=== AUGMENTATION ABLATION (cached-embedding subset, seed 42) ===")
for r in aug_rows:
    print("  %-20s n=%3d macroF1=%.3f (Δ%+.3f) MCC=%.3f" % (r["variant"], r["n_train"], r["macro_f1"], r["d_macro_f1_vs_full"], r["mcc"]))

2026-08-08 21:43:29,214 | INFO | consolidated: routing/gate/fusion/loss (3.3) + backbone (2.3) + token-cap (2.3) + augmentation (new) + efficiency (3.3)


=== AUGMENTATION ABLATION (cached-embedding subset, seed 42) ===
  original_only        n=1846 macroF1=0.551 (Δ+0.059) MCC=0.532
  +CSS                 n=1897 macroF1=0.487 (Δ-0.005) MCC=0.468
  +EDA                 n=1930 macroF1=0.444 (Δ-0.048) MCC=0.428
  +generation (full)   n=1971 macroF1=0.492 (Δ+0.000) MCC=0.457


## 7. Persist validation_report.json (consolidated)

In [8]:
validation_report = {
    "part": "stage5_part5.2_classification_generalization_ablation", "version": PART_VERSION,
    "principle": "P1; P3 (train-YT/test-Reddit+Twitter = cross-platform generalization); P6 (ablation+efficiency); "
                 " classification suite;  generalization;  ablation battery;  efficiency",
    "seeds": SEEDS,
    "classification_suite_ci": classification,
    "cross_platform_generalization": generalization,
    "ablation_battery": ablation_battery,
    "efficiency": efficiency,
    "benchmarking_caveat": "Cross-study F1/accuracy values are NOT directly comparable across papers — different "
                           "datasets, label schemes, splits, and protocols (Bhatt 2026; Garg 2023). We compare only "
                           "within this project's fixed frozen-TEST protocol.",
    "honest_note": "Pilot-scale: classification CIs bound SEED stability; the in-domain(%.3f)->cross-platform(%.3f) "
                   "gap is the headline generalization finding; Crisis recall ~0 carries through (13 test posts). "
                   "Ablation rows REUSE prior reports (cited) except the augmentation rows, computed here." % (
                       in_domain_f1, cross_f1),
}
(OUT_DIR / "validation_report.json").write_text(json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8")
log.info("wrote validation_report.json")

2026-08-08 21:43:29,312 | INFO | wrote validation_report.json


## 8. Figures — consolidated confusion (mean) + ablation battery

In [9]:
if HAS_MPL:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    # mean confusion
    cm = np.array(classification["mean_confusion_matrix"])
    fig, ax = plt.subplots(figsize=(4.6, 4.0)); im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4)); ax.set_yticks(range(4))
    ax.set_xticklabels([TIER_NAMES[t] for t in range(4)], rotation=30, ha="right", fontsize=8)
    ax.set_yticklabels([TIER_NAMES[t] for t in range(4)], fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Mean confusion (5 seeds, frozen TEST)", fontsize=9)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, "%.0f" % cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT_DIR / "confusion_consolidated.png", dpi=130); plt.close(fig)
    # ablation battery: architecture deltas + augmentation
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    if abl:
        groups = []
        for axis in ("routing", "gate", "fusion", "loss"):
            for r in abl["ablations"].get(axis, []):
                groups.append(("%s:%s" % (axis[:4], str(r["value"])[:10]), r.get("d_macro_f1_vs_baseline", 0.0)))
        labels = [g[0] for g in groups]; vals = [g[1] for g in groups]
        ax1.barh(range(len(vals)), vals, color=["#1b7837" if v >= 0 else "#b2182b" for v in vals])
        ax1.set_yticks(range(len(labels))); ax1.set_yticklabels(labels, fontsize=7)
        ax1.axvline(0, color="k", lw=0.6); ax1.set_xlabel("Δ macro-F1 vs baseline"); ax1.set_title("Architecture ablations (3.3)", fontsize=9)
    av = [r["macro_f1"] for r in aug_rows]; an = [r["variant"] for r in aug_rows]
    ax2.plot(range(len(av)), av, "o-", color="#1b7837")
    ax2.set_xticks(range(len(an))); ax2.set_xticklabels(an, rotation=20, ha="right", fontsize=7)
    ax2.set_ylabel("macro-F1 (frozen TEST)"); ax2.set_title("Augmentation ablation (new)", fontsize=9)
    fig.tight_layout(); fig.savefig(OUT_DIR / "ablation_battery.png", dpi=130); plt.close(fig)
    log.info("wrote confusion_consolidated.png + ablation_battery.png")
else:
    log.warning("matplotlib absent — skipped figures")

2026-08-08 21:43:31,892 | INFO | wrote confusion_consolidated.png + ablation_battery.png


## 9. This step is complete when

**notebook 15 is complete when:** the **classification suite** (per-tier P/R/F1 + mean confusion + MCC + ROC-AUC) is
reported **as mean ± 95 % CI** over the seeds; the **train-YouTube / test-Reddit+Twitter** cross-platform
generalization result is first-class (per-platform ± CI + the in-domain↔cross gap); the **consolidated ablation
battery** (routing/gate/fusion/loss reused from 3.3 + backbone/token-cap reused from 2.3 + augmentation computed
here) and the **efficiency** table exist with every reused row cited; the **benchmarking caveat** is written.
`validation_report.json` + figures produced. **No human; frozen TEST honoured; P6 satisfied.**

In [10]:
dod = all([
    (OUT_DIR / "validation_report.json").exists(),
    classification["overall_ci"]["macro_f1"] is not None,
    "per_platform_ci" in generalization,
    len(aug_rows) == 4,
    ablation_battery["architecture_3.3"]["fusion"] is not None,
])
print("=" * 64); print("Objective 5 · notebook 15 — VALIDATION CONSOLIDATION CARD"); print("=" * 64)
c = classification["overall_ci"]
print("classification : macroF1 %.3f [%.3f,%.3f] | MCC %.3f | ROC-AUC %s (5-seed CI)" % (
    c["macro_f1"]["mean"], c["macro_f1"]["ci95_lo"], c["macro_f1"]["ci95_hi"], c["mcc"]["mean"],
    c["roc_auc"]["mean"] if c["roc_auc"] else None))
print("generalization : in-domain %.3f -> cross-platform %.3f (gap %.3f)" % (
    in_domain_f1, cross_f1, generalization["generalization_gap_macro_f1"]))
print("ablation battery: routing/gate/fusion/loss (3.3) + backbone/token-cap (2.3) + augmentation (new, %d rows)" % len(aug_rows))
print("efficiency     : %s params head | infer %s ms/sample" % (efficiency["trainable_params_head"], efficiency["infer_ms_per_sample"]))
print("\nCompletion check:", "PASS" if dod else "INCOMPLETE", "(consolidated; magnitudes provisional, pilot-scale)")

Objective 5 · notebook 15 — VALIDATION CONSOLIDATION CARD
classification : macroF1 0.465 [0.434,0.495] | MCC 0.429 | ROC-AUC 0.8271 (5-seed CI)
generalization : in-domain 0.834 -> cross-platform 0.465 (gap 0.369)
ablation battery: routing/gate/fusion/loss (3.3) + backbone/token-cap (2.3) + augmentation (new, 4 rows)
efficiency     : 805572 params head | infer 0.1155 ms/sample

Completion check: PASS (consolidated; magnitudes provisional, pilot-scale)
